# Der Dehnstab – Verfahren nach Ritz

<div style="border: 2px solid #4bb98c; padding: 15px; border-radius: 8px; background-color: #f0faf6; margin-bottom: 15px;">
    <h3 style="margin-top:0; color:#2d7a5c;">Lernziele</h3>
    <ol>
        <li>Die <strong>schwache Form</strong> des Dehnstabs kennen und interpretieren können</li>
        <li>Das <strong>Ritz-Verfahren</strong> als systematischen Näherungsansatz verstehen</li>
        <li>Einen <strong>Polynomansatz</strong> wählen, einsetzen und das resultierende Gleichungssystem aufstellen</li>
        <li>Ergebnisse mit der <strong>analytischen Lösung</strong> vergleichen und die Güte der Näherung beurteilen</li>
    </ol>
</div>

<div style="border: 1px solid #ccc; padding: 10px; border-radius: 5px; background-color: #f9f9f9;">
    <p>
        Das <strong>Ritz-Verfahren</strong> wurde von <strong>Walther Ritz</strong> (1908) entwickelt und dient zur 
        näherungsweisen Lösung von Randwertproblemen. Die Grundidee: Man setzt die unbekannte Lösung als 
        <strong>Linearkombination gewählter Ansatzfunktionen</strong> an und bestimmt die freien Koeffizienten so, 
        dass die schwache Form erfüllt wird.
        Es gilt als <strong>Vorgängermethode der Finite-Elemente-Methode (FEM)</strong>.
    </p>
</div>

## 1. Problemstellung

Wir betrachten einen eingespannten Dehnstab der Länge $\ell$ mit:
- konstanter Dehnsteifigkeit $EA$
- konstanter Streckenlast $n$ (z.B. Eigengewicht)
- einer Einzelkraft $F$ am freien Ende

<img src="images/Stab.png" width="25%">

**Differentialgleichung** (starke Form):

$$
EA \, u''(x) = -n(x)
$$

**Randbedingungen:**
- Links eingespannt: $u(0) = 0$ (Dirichlet)  
- Rechts belastet: $EA \, u'(\ell) = F$ (Neumann)

**Schwache Form** (aus der Vorlesung bekannt):

$$
\underbrace{\int_{0}^{\ell} \delta \varepsilon \cdot E \varepsilon \, A \; \mathrm{d}x}_{\text{innere virtuelle Arbeit}} 
- \underbrace{\int_0^{\ell} \delta u \cdot n \; \mathrm{d}x}_{\text{virt. Arbeit der Streckenlast}} 
- \underbrace{\delta u(\ell) \cdot F}_{\text{virt. Arbeit der Einzelkraft}} = 0
$$

mit der Dehnung $\varepsilon = u'(x)$ und der virtuellen Dehnung $\delta\varepsilon = \delta u'(x)$.

<div style="border-left: 4px solid #f0ad4e; padding: 10px; background-color: #fefbe8; margin: 10px 0;">
    <strong>Reflexionsfrage:</strong> Welche Stetigkeitsanforderungen stellt die schwache Form an die Ansatzfunktion $u_h(x)$? 
    <em>(Hinweis: Welche Ableitungen treten auf?)</em>
</div>

> In der schwachen Form kommt höchstens die **erste Ableitung** $u'(x)$ vor. Daher muss $u_h(x)$ **stetig** und **stückweise differenzierbar** sein (mathematisch: $u_h \in H^1$).

## 2. Ansatzfunktion wählen

Die Idee des Ritz-Verfahrens: Wir setzen die unbekannte Verschiebung $u(x)$ als **Polynom** an:

$$
u_h(x) = \sum_{i=0}^{N} a_i \, x^i = a_0 + a_1 \, x + a_2 \, x^2 + \ldots
$$

Die Koeffizienten $a_i$ sind die **unbekannten Freiheitsgrade**, die wir bestimmen wollen.

Für die virtuelle Verschiebung $\delta u$ wählen wir denselben Ansatzraum:

$$
\delta u_h(x) = \sum_{i=0}^{N} \delta a_i \, x^i
$$

<div style="border-left: 4px solid #5bc0de; padding: 10px; background-color: #eef7fb; margin: 10px 0;">
    <strong>Hinweis:</strong> Starten Sie mit <code>num_par = 3</code> (d.h. $a_0, a_1, a_2$). 
    Experimentieren Sie später mit <code>num_par = 2</code> oder <code>num_par = 4</code>, 
    um den Einfluss der Ansatzordnung zu beobachten!
</div>

In [63]:
import sympy as sp
from IPython.display import display, Math, Latex

# Freie Variable
x = sp.symbols('x')

# ============================================================
#  PARAMETER: Anzahl der Ansatzkoeffizienten (hier variieren!)
# ============================================================
num_par = 3  # -> Ansatz: a_0 + a_1*x + a_2*x^2

# Unbekannte Koeffizienten a_i
alist = [sp.Symbol(f'a_{i}') for i in range(num_par)]
a = sp.Matrix(alist)

# Verschiebungsansatz u_h(x)
u_h = sum(a[i] * x**i for i in range(num_par))

display(Math(r'u_h(x) = ' + sp.latex(u_h)))

<IPython.core.display.Math object>

In [64]:
# Virtuelle Koeffizienten delta_a_i
dalist = [sp.Symbol(rf'\delta a_{i}') for i in range(num_par)]
da = sp.Matrix(dalist)

# Virtuelle Verschiebung delta_u(x)
delta_u = sum(da[i] * x**i for i in range(num_par))

display(Math(r'\delta u(x) = ' + sp.latex(delta_u)))

<IPython.core.display.Math object>

### Ableitungen bilden

Für die schwache Form benötigen wir die **Dehnungen**:

$$
\varepsilon = \frac{\mathrm{d} u_h}{\mathrm{d} x}, \qquad
\delta\varepsilon = \frac{\mathrm{d} \,\delta u}{\mathrm{d} x}
$$

In [65]:
epsilon = sp.diff(u_h, x)
delta_epsilon = sp.diff(delta_u, x)

display(Math(r'\varepsilon(x) = ' + sp.latex(epsilon)))
display(Math(r'\delta\varepsilon(x) = ' + sp.latex(delta_epsilon)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. Einsetzen in die schwache Form

Nun setzen wir die Ansätze in die drei Terme der schwachen Form ein.

Wir definieren zunächst die **symbolischen Materialparameter**:

In [66]:
# Symbolische Parameter (werden erst später durch Zahlenwerte ersetzt)
E_s, A_s, F_s, n_s, ell = sp.symbols('E A F n ell')

### Term 1: Innere virtuelle Arbeit (Formänderungsarbeit)

$$
W_{\text{int}} = \int_0^{\ell} \delta\varepsilon \cdot E \, A \cdot \varepsilon \; \mathrm{d}x
$$

In [69]:
W_int = sp.simplify(
    sp.integrate(
        delta_epsilon * E_s * A_s * epsilon
        , (x, 0, ell))
    )

display(Math(r'W_{\mathrm{int}} = ' + sp.latex(W_int)))

<IPython.core.display.Math object>

### Term 2: Virtuelle Arbeit der Einzelkraft

$$
W_F = \delta u(\ell) \cdot F
$$

In [72]:
W_F = delta_u.subs(x, ell) * F_s

display(Math(r'W_F = ' + sp.latex(W_F)))

<IPython.core.display.Math object>

### Term 3: Virtuelle Arbeit der Streckenlast

$$
W_n = \int_0^{\ell} \delta u \cdot n \; \mathrm{d}x
$$

In [70]:
W_n = sp.simplify(
    sp.integrate(
        delta_u * n_s,
        (x, 0, ell))
    )

display(Math(r'W_n = ' + sp.latex(W_n)))

<IPython.core.display.Math object>

### Schwache Form zusammensetzen

$$
W_{\text{int}} - W_F - W_n = 0
$$

In [73]:
schwache_Form = sp.Eq(W_int - W_F - W_n, 0)

display(schwache_Form)

Eq(A*E*ell*(3*\delta a_1*a_1 + 4*\delta a_2*a_2*ell**2 + 3*ell*(\delta a_1*a_2 + \delta a_2*a_1))/3 - F*(\delta a_0 + \delta a_1*ell + \delta a_2*ell**2) - ell*n*(6*\delta a_0 + 3*\delta a_1*ell + 2*\delta a_2*ell**2)/6, 0)

## 4. Gleichungssystem aufstellen

<div style="border-left: 4px solid #5bc0de; padding: 10px; background-color: #eef7fb; margin: 10px 0;">

<strong>Schlüsselidee:</strong> 
Die schwache Form muss für <em>beliebige</em> virtuelle Verschiebungen $\delta a_i$ gelten. Daraus folgt, dass alle Koeffizienten der $\delta a_i$ einzeln null sein müssen.

Daher können wir die Gleichung nach den $\delta a_i$ „sortieren" und erhalten ein <strong>lineares Gleichungssystem</strong>:

$$\mathbf{K} \cdot \mathbf{a} = \mathbf{f}$$

mit der <strong>Steifigkeitsmatrix</strong> $\mathbf{K}$ und dem <strong>Lastvektor</strong> $\mathbf{f}$.
</div>

Die Einträge werden durch Koeffizientenvergleich extrahiert:

$$
K_{ij} = \frac{\partial^2 (\text{schwache Form})}{\partial\, \delta a_i \;\partial\, a_j}
\qquad\text{und}\qquad
f_i = -\frac{\partial (\text{schwache Form})}{\partial\, \delta a_i}\bigg|_{\mathbf{a}=\mathbf{0}}
$$

In [75]:
n_eq = len(dalist)  # Anzahl Gleichungen
m_eq = len(alist)   # Anzahl Unbekannte

# Steifigkeitsmatrix K
K = sp.zeros(n_eq, m_eq)
# Lastvektor f
f = sp.zeros(m_eq, 1)

for i in range(n_eq):
    for j in range(m_eq):
        K[i, j] = sp.diff(schwache_Form.lhs, dalist[i], alist[j])
    # Lastvektor: Terme ohne a_j (mit negativem Vorzeichen, da nach rechts gebracht)
    f[i, 0] = -schwache_Form.lhs.subs({alist[j]: 0 for j in range(m_eq)}).diff(dalist[i])

print("Steifigkeitsmatrix K:")
display(K)
print("\nLastvektor f:")
display(f)

Steifigkeitsmatrix K:


Matrix([
[0,          0,              0],
[0,    A*E*ell,     A*E*ell**2],
[0, A*E*ell**2, 4*A*E*ell**3/3]])


Lastvektor f:


Matrix([
[            F + ell*n],
[   F*ell + ell**2*n/2],
[F*ell**2 + ell**3*n/3]])

## 5. Randbedingungen einarbeiten und lösen

Die Dirichlet-Randbedingung $u(0) = 0$ liefert eine **zusätzliche Gleichung**. Wir lösen das Gesamtsystem nach den Koeffizienten $a_i$:

In [76]:
# Randbedingung u(0) = 0 und Gleichungssystem K*a = f
solution = sp.solve([K * a - f, u_h.subs(x, 0)], alist)

print("L\u00f6sung f\u00fcr die Koeffizienten:")
for key, val in solution.items():
    display(Math(f'{sp.latex(key)} = {sp.latex(val)}'))

Lösung für die Koeffizienten:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Verschiebungsfeld $u_h(x)$

In [77]:
u_ritz = sp.collect(sp.expand(u_h.subs(solution)), x)

display(Math(r'u_h(x) = ' + sp.latex(u_ritz)))

<IPython.core.display.Math object>

## 6. Numerische Auswertung und Vergleich

Jetzt setzen wir **konkrete Zahlenwerte** ein und vergleichen mit der **analytischen Lösung**.

| Parameter | Wert | Einheit | Beschreibung |
|:---------:|:----:|:-------:|:-------------|
| $E$       | 210 000 | N/mm² | E-Modul (Stahl) |
| $A$       | 400 | mm² | Querschnittsfläche (20 × 20 mm) |
| $\ell$    | 3 000 | mm | Stablänge |
| $n$       | 100 | N/mm | Streckenlast |
| $F$       | 10 000 | N | Einzelkraft am rechten Ende |

**Analytische Lösung** (durch direktes Integrieren der DGL):

$$
u_{\text{exakt}}(x) = \frac{n}{2EA} x^2 - \frac{F + n\ell}{EA} x
$$

In [79]:
import numpy as np
import plotly.graph_objects as go

# Zahlenwerte als Dictionary (Schlüssel = SymPy-Symbole!)
param = {
    E_s: 210_000,   # N/mm²
    A_s: 400,       # mm²
    ell: 3_000,     # mm
    n_s: 100,       # N/mm
    F_s: 10_000     # N
}

# Ritz-Lösung numerisch auswerten
u_ritz_fun = sp.lambdify(x, u_ritz.subs(param), 'numpy')

# Analytische Lösung
E_val, A_val, ell_val = param[E_s], param[A_s], param[ell]
n_val, F_val = param[n_s], param[F_s]

u_analytic = (n_val / (2 * E_val * A_val)) * x**2 - (F_val + n_val * ell_val) / (E_val * A_val) * x
u_analytic_fun = sp.lambdify(x, -u_analytic, 'numpy')

# Auswertungspunkte
x_plot = np.linspace(0, ell_val, 200)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_plot, y=u_ritz_fun(x_plot),
    mode='lines', name='Ritz-Lösung',
    line=dict(color='#1f77b4', width=3)
))
fig.add_trace(go.Scatter(
    x=x_plot, y=u_analytic_fun(x_plot),
    mode='lines', name='Analytische Lösung',
    line=dict(color='#ff7f0e', width=2, dash='dash')
))
fig.update_layout(
    title='Verschiebung u(x) – Ritz vs. Analytisch',
    xaxis_title='Position x [mm]',
    yaxis_title='Verschiebung u [mm]',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
    font=dict(size=13)
)
fig.show()

<div style="border-left: 4px solid #5cb85c; padding: 10px; background-color: #eef9ee; margin: 10px 0;">
<strong>Beobachtung:</strong> Bei <code>num_par = 3</code> (quadratischer Ansatz) stimmen Ritz-Lösung und analytische Lösung 
<strong>exakt</strong> überein! Das liegt daran, dass die analytische Lösung selbst ein Polynom 2. Grades ist 
und somit im gewählten Ansatzraum enthalten ist.
</div>

<div style="border-left: 4px solid #f0ad4e; padding: 10px; background-color: #fefbe8; margin: 10px 0;">
    <strong>Aufgabe:</strong> Ändern Sie <code>num_par = 2</code> (linearer Ansatz). Was beobachten Sie? 
    Warum weicht die Lösung ab?
</div>

## 7. Spannungsverteilung

Die Normalspannung ergibt sich aus dem Materialgesetz (Hookesches Gesetz):

$$
\sigma(x) = E \cdot \varepsilon(x) = E \cdot u_h'(x)
$$

<div style="border-left: 4px solid #d9534f; padding: 10px; background-color: #fdf0ef; margin: 10px 0;">
<strong>Wichtig:</strong> Da die Spannung von der <strong>Ableitung</strong> der Verschiebung abhängt, ist sie 
um einen Polynomgrad niedriger als $u_h(x)$. Bei einem linearen Ansatz (<code>num_par = 2</code>) ist die 
Spannung daher <strong>konstant</strong> – was physikalisch bei Eigengewichtsbelastung nicht korrekt ist!
</div>

In [81]:
# Spannung aus Ritz-Lösung
sigma_ritz = E_s * sp.diff(u_ritz, x)
sigma_ritz_sub = sigma_ritz.subs(param)

display(Math(r'\sigma_{\mathrm{Ritz}}(x) = ' + sp.latex(sigma_ritz)))

# Numerische Auswertung (Schutz vor skalarem Ergebnis bei konstantem sigma)
sig_ritz_fun = sp.lambdify(x, sigma_ritz_sub, 'numpy')

def sigma_safe(x_arr):
    result = sig_ritz_fun(x_arr)
    if np.isscalar(result):
        result = np.full_like(x_arr, result, dtype=float)
    return result

# Analytische Spannung
sigma_analytic = E_val * sp.diff(-u_analytic, x)
sig_analytic_fun = sp.lambdify(x, sigma_analytic, 'numpy')

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_plot, y=sigma_safe(x_plot),
    mode='lines', name='σ(x) – Ritz-Lösung',
    line=dict(color='#1f77b4', width=3)
))
fig.add_trace(go.Scatter(
    x=x_plot, y=sig_analytic_fun(x_plot),
    mode='lines', name='σ(x) – Analytisch',
    line=dict(color='#ff7f0e', width=2, dash='dash')
))
fig.update_layout(
    title='Spannungsverteilung σ(x) – Ritz vs. Analytisch',
    xaxis_title='Position x [mm]',
    yaxis_title='Spannung σ [N/mm²]',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
    font=dict(size=13)
)
fig.show()

<IPython.core.display.Math object>

## 8. Fehlerbetrachtung

Um die **Qualität der Näherung** quantitativ zu bewerten, berechnen wir den relativen Fehler 
der Verschiebung am rechten Stabende:

In [82]:
# Verschiebung am rechten Ende
u_ritz_end = float(u_ritz.subs(param).subs(x, ell_val))
u_exact_end = float(u_analytic.subs(x, ell_val))

print(f"Verschiebung am rechten Ende (x = {ell_val} mm):")
print(f"  Ritz-Lösung:       u_Ritz(l)  = {u_ritz_end:.6f} mm")
print(f"  Analytische Lsg.:  u_exakt(l) = {u_exact_end:.6f} mm")
if abs(u_exact_end) > 0:
    print(f"  Relativer Fehler:  {abs(u_ritz_end - u_exact_end) / abs(u_exact_end) * 100:.4f} %")
else:
    print(f"  Absoluter Fehler:  {abs(u_ritz_end - u_exact_end):.6e} mm")

Verschiebung am rechten Ende (x = 3000 mm):
  Ritz-Lösung:       u_Ritz(l)  = 5.714286 mm
  Analytische Lsg.:  u_exakt(l) = -5.714286 mm
  Relativer Fehler:  200.0000 %


## 9. Zusammenfassung

<div style="border: 2px solid #4bb98c; padding: 15px; border-radius: 8px; background-color: #f0faf6;">

| Schritt | Was passiert? |
|:-------:|:-------------|
| **1** | Ansatzfunktion $u_h(x) = \sum a_i x^i$ wählen |
| **2** | Ansatz und seine Ableitung in die **schwache Form** einsetzen |
| **3** | Durch Koeffizientenvergleich bzgl. $\delta a_i$ ein **lineares Gleichungssystem** $\mathbf{K}\mathbf{a} = \mathbf{f}$ gewinnen |
| **4** | Randbedingungen einarbeiten und nach $\mathbf{a}$ lösen |
| **5** | Ergebnis auswerten und mit **analytischer Lösung** vergleichen |

**Kernaussage:** Das Ritz-Verfahren ist exakt, wenn die **analytische Lösung im Ansatzraum** enthalten ist. Andernfalls liefert es die **beste Näherung** im Sinne der Energienorm.

</div>

---

## Übungsaufgaben

<div style="border: 1px solid #337ab7; padding: 15px; border-radius: 5px; background-color: #f5f9fc; margin: 10px 0;">

**Aufgabe 1 – Einfluss der Ansatzordnung:**  
Setzen Sie <code>num_par = 2</code> (linearer Ansatz). Vergleichen Sie Verschiebung und Spannung mit der analytischen Lösung. Warum ist die Spannung jetzt konstant?


**Aufgabe 2 – Höherer Ansatz:**  
Setzen Sie <code>num_par = 4</code>. Ändert sich die Lösung? Begründen Sie Ihre Beobachtung.

</div>